# Divergence investigation

Compare pairwise **divergence** across GBOV + GoldenSites (2023 / 2024):

| column | meaning |
|--------|---------|
| `gvf_vs_gcc_div` | satellite GVF vs PhenoCam GCC |
| `gvf_vs_ndvi_div` | satellite GVF vs PhenoCam NDVI |
| `gcc_vs_ndvi_div` | PhenoCam GCC vs PhenoCam NDVI (ground cross-check) |

Divergence is the combined gap+DTW score used elsewhere (lower = better agreement).

Reads existing `anomaly_pipeline/output/metadata/*_scores.csv` (run
`anomaly_check.ipynb` score cells first if those are missing).

**Spin-up** (`gvf_sos == 1`) is dropped by default before plots/tables.

Main figure: grouped bar chart of mean divergence by veg.
Artifacts: `anomaly_pipeline/output/divergence/`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import load_all_scores, load_table

ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"
OUT_DIR = ANOMALY_DIR / "divergence"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIV_COLS = ["gvf_vs_gcc_div", "gvf_vs_ndvi_div", "gcc_vs_ndvi_div"]
DIV_LABELS = {
    "gvf_vs_gcc_div": "GVF vs GCC",
    "gvf_vs_ndvi_div": "GVF vs NDVI",
    "gcc_vs_ndvi_div": "GCC vs NDVI",
}
SOURCES = [
    "GBOV_2023",
    "GBOV_2024",
    "GoldenSites_2023",
    "GoldenSites_2024",
]


## Load scores

Concatenate all metadata score tables. Flag / drop spin-up.


In [ ]:
all_scores = load_all_scores(ANOMALY_DIR)
all_scores = all_scores.loc[all_scores["source"].isin(SOURCES)].copy()
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0) if "gvf_sos" in all_scores.columns else False

EXCLUDE_SPINUP = True
pool = all_scores.loc[~all_scores["spin_up"]].copy() if EXCLUDE_SPINUP else all_scores.copy()

print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"pool={len(pool)} | sources={sorted(pool['source'].unique())}"
)
print("missing divergence counts:")
display(pool[DIV_COLS].isna().sum().to_frame("n_missing"))

show_cols = [
    c for c in [
        "site", "veg", "year", "source",
        *DIV_COLS,
        "gvf_vs_gcc_gap", "gvf_vs_ndvi_gap", "gcc_vs_ndvi_gap",
        "gvf_sos", "spin_up",
    ] if c in pool.columns
]
display(pool[show_cols].head(12))


## Summary tables

Mean / median divergence by **source** and by **veg** (within the clean pool).


In [ ]:
def summarize(df: pd.DataFrame, by: str) -> pd.DataFrame:
    cols = [c for c in DIV_COLS if c in df.columns]
    g = df.groupby(by, dropna=False)[cols]
    out = g.agg(["count", "mean", "median"]).round(3)
    # flatten MultiIndex columns
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    return out.reset_index()

by_source = summarize(pool, "source")
by_veg = summarize(pool, "veg")
by_source_veg = (
    pool.groupby(["source", "veg"], dropna=False)[DIV_COLS]
    .mean()
    .round(3)
    .reset_index()
)

print("By source:")
display(by_source)
print("By veg:")
display(by_veg)
print("By source × veg (mean):")
display(by_source_veg)

by_source.to_csv(OUT_DIR / "divergence_by_source.csv", index=False)
by_veg.to_csv(OUT_DIR / "divergence_by_veg.csv", index=False)
by_source_veg.to_csv(OUT_DIR / "divergence_by_source_veg.csv", index=False)
print(f"Wrote summary CSVs under {OUT_DIR}")


## Grouped bars: mean divergence by veg

For each veg type, three bars = GVF–GCC / GVF–NDVI / GCC–NDVI means.


In [ ]:
def plot_div_bars_by_veg(df: pd.DataFrame, out_png: Path) -> Path:
    vegs = sorted(df["veg"].dropna().unique())
    x = np.arange(len(vegs))
    width = 0.25
    colors = ["#4C78A8", "#F58518", "#54A24B"]

    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(vegs) + 3), 4.8))
    for i, (col, color) in enumerate(zip(DIV_COLS, colors)):
        means = [
            df.loc[df["veg"].eq(v), col].mean(skipna=True)
            for v in vegs
        ]
        ax.bar(x + (i - 1) * width, means, width, label=DIV_LABELS[col], color=color)

    counts = [int(df["veg"].eq(v).sum()) for v in vegs]
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v}\n(n={n})" for v, n in zip(vegs, counts)])
    ax.set_ylabel("mean divergence")
    ax.set_xlabel("veg")
    ax.set_title("Mean pairwise divergence by veg (all sources)")
    ax.legend(frameon=True)
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {out_png}")
    return out_png

plot_div_bars_by_veg(pool, OUT_DIR / "divergence_bars_by_veg.png")


## Notes

- Lower divergence = better agreement.
- `gcc_vs_ndvi_div` is the PhenoCam-only baseline; large values mean GCC and NDVI
  already disagree on the ground, so GVF mismatches are harder to interpret.
- Toggle `EXCLUDE_SPINUP` in the load cell if you want spin-up rows included.
